In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

# Search for the uploaded file in Google Drive
matches = glob.glob("/content/drive/MyDrive/**/news_raw_basketA.csv", recursive=True)

if not matches:
    raise FileNotFoundError(
        "Could not find 'news_raw_basketA.csv' in Google Drive. "
        "Please check if Google Drive is mounted or if the file name/path is correct."
    )

SRC = matches[0]
OUT = SRC.replace("news_raw_basketA.csv", "news_scores_basketA.csv")
print("reading:", SRC)

In [ ]:
df = pd.read_csv(SRC, low_memory=False); df["row_id"] = df.index

In [ ]:
for name in ["out", "df", "scores", "res"]:
    if name in globals():
        print(name, globals()[name].shape, globals()[name].columns.tolist())

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Device check
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

# 2. Model & Tokenizer loading
tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(dev).eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
pos_i = [k for k, v in id2label.items() if v == "positive"][0]
neg_i = [k for k, v in id2label.items() if v == "negative"][0]

texts = df["Article_title"].fillna("").astype(str).tolist()

# 3. Fast inference setup
BATCH = 128 if dev == "cuda" else 32
scores = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH):
        batch_texts = texts[i:i+BATCH]
        enc = tok(batch_texts, padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(dev)

        # Use mixed precision on GPU for speed
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**enc).logits

        p = torch.softmax(logits, dim=1).cpu().numpy()
        scores.extend((p[:, pos_i] - p[:, neg_i]).tolist())

        if (i // BATCH) % 50 == 0:
            print(f"Processed {i:,} / {len(texts):,}", flush=True)

# 4. Save output
df["sentiment_score"] = scores
out = df[["row_id", "Date", "Stock_symbol", "sentiment_score"]]
out.to_csv("/content/drive/MyDrive/us_experiment/news_scores_basketA.csv", index=False)

print("Finished! Output shape:", out.shape)

In [ ]:
print(out["sentiment_score"].describe())
print((out["sentiment_score"] > 0.3).mean(), (out["sentiment_score"] < -0.3).mean())
print(df.sample(8, random_state=1)[["Stock_symbol", "Article_title", "sentiment_score"]].to_string())